# 📚 מחברת 1: ויקיפדיה כמקור נתונים ורשתות ידע
## מבוא למדעי הרוח הדיגיטליים | אוניברסיטת אריאל | תשפ"ו
### פרופ' שי גורדין
---
**מה נלמד במחברת זו?**

במחברת זו נלמד כיצד ויקיפדיה משמשת כמקור נתונים עבור מחקר מדעי הרוח, ולבנות רשת ידע אינטראקטיבית.

**נושאים:**
1. ויקיפדיה כמקור מידע – יתרונות ומגבלות
2. ה-API של מדיה-ויקי – גישה תכנותית
3. שליפת נתונים מקטגוריות עבריות
4. בניית רשת ידע עם NetworkX
5. ויזואליזציה אינטראקטיבית עם PyVis
6. ניתוח פערים – ערכים חסרים = פרויקטי סיום!

**🎯 הקטגוריות שנחקור:**
- 🏺 ארכיאולוגיה של ארץ ישראל
- 📜 היסטוריה של עם ישראל
- 💻 מדעי הרוח הדיגיטליים

## חלק א: ויקיפדיה כמקור נתונים מחקרי

### מדוע ויקיפדיה מעניינת את מדעי הרוח הדיגיטליים?

ויקיפדיה אינה רק אנציקלופדיה – היא **מסד נתונים ענק** של ידע אנושי מובנה:

| מאפיין | פרטים |
|--------|--------|
| **היקף** | ~6.7M ערכים באנגלית, ~380K בעברית |
| **קישוריות** | כל ערך מקושר לאחרים = רשת ידע |
| **קטגוריות** | ארגון היררכי של ידע |
| **היסטוריית עריכות** | ניתן לחקור התפתחות ידע לאורך זמן |
| **API חינמי** | גישה פרוגרמטית מלאה |

### שאלות מחקריות:
- אילו נושאים **מיוצגים-חסר** (underrepresented)?
- כיצד **מתפתח הידע** לאורך זמן?
- מהי **הרשת** של קשרים בין מושגים?
- מה ה**"ליבה"** של שדה ידע?

> 📖 לקריאה: Niederer & van Dijck (2010). *Wisdom of the crowd or technicity of content?* New Media & Society.

In [ ]:
# התקנת ספריות נדרשות
!pip install pyvis wikipedia-api tqdm -q

print("✅ הספריות הותקנו!")
print("   • pyvis      - ויזואליזציה אינטראקטיבית")
print("   • wikipedia-api - גישה ל-API")
print("   • tqdm       - סרגל התקדמות")

In [ ]:
# יבוא ספריות
import requests          # שליפת נתונים מהרשת
import json              # עיבוד JSON
import pandas as pd      # ניהול נתונים בטבלאות
import networkx as nx    # בניית ועיבוד גרפים
from pyvis.network import Network   # ויזואליזציה אינטראקטיבית
import time              # עיכובים בין בקשות
from tqdm.notebook import tqdm      # סרגל התקדמות
from IPython.display import HTML, display
import matplotlib.pyplot as plt
import matplotlib
import os

matplotlib.rcParams['axes.unicode_minus'] = False

print("✅ כל הספריות יובאו!")
print(f"   NetworkX {nx.__version__} | Pandas {pd.__version__}")

## חלק ב: ה-API של ויקיפדיה

### מהו API?
**API** (Application Programming Interface) = ממשק תכנות יישומים.  
חשבו עליו כ"תפריט" – אנחנו בוחרים מה לבקש, ה-API מחזיר את הנתונים.

### כתובת הבסיס של ויקיפדיה העברית:
```
https://he.wikipedia.org/w/api.php
```

### דוגמה לבקשה:
```
https://he.wikipedia.org/w/api.php?action=query&list=categorymembers&cmtitle=קטגוריה:ארכיאולוגיה_של_ארץ_ישראל&format=json
```

### פרמטרים עיקריים:
| פרמטר | תיאור |
|--------|-------|
| `action=query` | שאילתת מידע |
| `list=categorymembers` | רשימת חברי קטגוריה |
| `prop=links` | קישורים בתוך ערך |
| `prop=extracts` | תוכן/תקציר הערך |
| `format=json` | פורמט התגובה |

📖 **תיעוד מלא**: https://www.mediawiki.org/wiki/API:Main_page

In [ ]:
# ============================================================
# מחלקת גישה לוויקיפדיה העברית
# ============================================================

class HebrewWikipedia:
    """
    ממשק לוויקיפדיה העברית דרך MediaWiki API.
    
    שימוש:
        wiki = HebrewWikipedia()
        pages = wiki.get_category_members("ארכיאולוגיה של ארץ ישראל")
        links = wiki.get_page_links("מגידו")
    """
    
    BASE_URL = "https://he.wikipedia.org/w/api.php"
    
    def __init__(self, delay=0.15):
        """
        אתחול הממשק.
        delay: עיכוב (שניות) בין בקשות – כיבוד מגבלות השרת
        """
        self.session = requests.Session()
        self.delay = delay
        # חשוב: זיהוי הבוט – כיבוד נוהלי ויקיפדיה
        self.session.headers.update({
            'User-Agent': 'IntroDH-Course-Bot/1.0 (Ariel University; Educational) '
                          'Contact: shygordin@gmail.com'
        })
    
    def _request(self, params):
        """שליחת בקשה ל-API עם טיפול בשגיאות. פונקציה פנימית."""
        params['format'] = 'json'
        try:
            resp = self.session.get(self.BASE_URL, params=params, timeout=30)
            resp.raise_for_status()
            time.sleep(self.delay)
            return resp.json()
        except Exception as e:
            print(f"⚠️ שגיאה: {e}")
            return {}
    
    def get_category_members(self, category_name, depth=1):
        """
        שליפת כל הערכים בקטגוריה (כולל תת-קטגוריות).
        
        פרמטרים:
            category_name : שם הקטגוריה ללא הקידומת "קטגוריה:"
            depth         : עומק חיפוש בתת-קטגוריות
        
        מחזיר: רשימת dict עם: title, pageid, category
        """
        pages = []
        seen_cats = set()
        
        def _fetch(cat, current_depth):
            if cat in seen_cats or current_depth > depth:
                return
            seen_cats.add(cat)
            
            continue_key = None
            while True:
                params = {
                    'action': 'query',
                    'list': 'categorymembers',
                    'cmtitle': f'קטגוריה:{cat}',
                    'cmlimit': 500,
                    'cmtype': 'page|subcat'
                }
                if continue_key:
                    params['cmcontinue'] = continue_key
                
                data = self._request(params)
                if 'query' not in data:
                    break
                
                for item in data['query']['categorymembers']:
                    if item['ns'] == 0:  # ערך רגיל
                        if not any(p['title'] == item['title'] for p in pages):
                            pages.append({
                                'title': item['title'],
                                'pageid': item['pageid'],
                                'category': category_name
                            })
                    elif item['ns'] == 14 and current_depth < depth:  # תת-קטגוריה
                        subcat = item['title'].replace('קטגוריה:', '')
                        _fetch(subcat, current_depth + 1)
                
                if 'continue' in data:
                    continue_key = data['continue']['cmcontinue']
                else:
                    break
        
        _fetch(category_name, 0)
        return pages
    
    def get_page_links(self, title, filter_titles=None):
        """
        שליפת קישורים פנימיים מתוך ערך.
        
        פרמטרים:
            title         : כותרת הערך
            filter_titles : אם סופק, מחזיר רק קישורים לכותרות אלו
        
        מחזיר: רשימת כותרות מקושרות
        """
        links = []
        continue_key = None
        
        while True:
            params = {
                'action': 'query',
                'prop': 'links',
                'titles': title,
                'pllimit': 500,
                'plnamespace': 0
            }
            if continue_key:
                params['plcontinue'] = continue_key
            
            data = self._request(params)
            if 'query' not in data:
                break
            
            for pid, pdata in data['query']['pages'].items():
                for link in pdata.get('links', []):
                    t = link['title']
                    if filter_titles is None or t in filter_titles:
                        links.append(t)
            
            if 'continue' in data:
                continue_key = data['continue']['plcontinue']
            else:
                break
        
        return links
    
    def get_page_summary(self, title):
        """שליפת תקציר ומידע בסיסי על ערך."""
        data = self._request({
            'action': 'query',
            'prop': 'extracts|info',
            'titles': title,
            'exintro': True,
            'exchars': 800,
            'inprop': 'url'
        })
        if 'query' not in data:
            return {}
        for pid, pdata in data['query']['pages'].items():
            if pid != '-1':
                return pdata
        return {}
    
    def check_exists(self, titles):
        """
        בדיקת קיום רשימת ערכים.
        מחזיר dict: {כותרת: True/False}
        עובד באצוות של 50.
        """
        results = {}
        for i in range(0, len(titles), 50):
            batch = titles[i:i+50]
            data = self._request({
                'action': 'query',
                'prop': 'info',
                'titles': '|'.join(batch)
            })
            if 'query' not in data:
                continue
            for pid, pdata in data['query']['pages'].items():
                exists = pid != '-1' and 'missing' not in pdata
                results[pdata['title']] = exists
        return results
    
    def get_red_links(self, title):
        """
        שליפת קישורים אדומים (ערכים מוזכרים אך לא קיימים).
        אלו הם פערי הידע בוויקיפדיה!
        """
        data = self._request({
            'action': 'parse',
            'page': title,
            'prop': 'links'
        })
        missing = []
        if 'parse' in data:
            for link in data['parse'].get('links', []):
                if link.get('ns') == 0 and 'exists' not in link:
                    m = link.get('*', '')
                    if m:
                        missing.append(m)
        return missing


# יצירת אובייקט ויקיפדיה
wiki = HebrewWikipedia(delay=0.15)
print("✅ ממשק ויקיפדיה מוכן!")
print()
print("🔍 בדיקת חיבור...")
test = wiki.get_category_members("ארכיאולוגיה של ארץ ישראל", depth=0)
print(f"   נמצאו {len(test)} ערכים בקטגוריה הראשית")
if test:
    print(f"   דוגמה: {test[0]['title']}")

In [ ]:
# ============================================================
# הגדרת הקטגוריות ונתוני הויזואליזציה
# ============================================================

CATEGORIES = {
    'ארכיאולוגיה של ארץ ישראל': {
        'color': '#e74c3c',
        'bg_color': '#fadbd8',
        'icon': '🏺',
        'group': 1
    },
    'היסטוריה של עם ישראל': {
        'color': '#2980b9',
        'bg_color': '#d6eaf8',
        'icon': '📜',
        'group': 2
    },
    'מדעי הרוח הדיגיטליים': {
        'color': '#27ae60',
        'bg_color': '#d5f5e3',
        'icon': '💻',
        'group': 3
    }
}

print("📋 קטגוריות מוגדרות:")
for cat, info in CATEGORIES.items():
    print(f"  {info['icon']}  {cat}")

In [ ]:
# ============================================================
# שליפת כל הערכים מוויקיפדיה
# ============================================================
# ⏱️ תהליך זה עשוי לקחת 3–8 דקות.
# אנחנו שולחים מאות בקשות לשרת ויקיפדיה.

print("📥 שולף ערכים מוויקיפדיה העברית...")
print("=" * 55)

all_pages = []
category_pages = {}

for cat_name, cat_info in CATEGORIES.items():
    print(f"\n{cat_info['icon']} שולף: {cat_name}")
    print("   (כולל תת-קטגוריות, עומק 1)")
    
    pages = wiki.get_category_members(cat_name, depth=1)
    category_pages[cat_name] = pages
    all_pages.extend(pages)
    
    print(f"   ✓ נמצאו {len(pages)} ערכים")

print("\n" + "=" * 55)
print(f"📊 סה\"כ: {len(all_pages)} ערכים")

# יצירת DataFrame
df = pd.DataFrame(all_pages).drop_duplicates('title').reset_index(drop=True)
print(f"   לאחר הסרת כפילויות: {len(df)} ערכים ייחודיים")
print()
display(df.head(8))

In [ ]:
# ============================================================
# ניתוח ראשוני: גרף התפלגות
# ============================================================

counts = df.groupby('category').size().sort_values(ascending=False)

print("📊 התפלגות לפי קטגוריה:")
for cat, n in counts.items():
    icon = CATEGORIES[cat]['icon']
    bar = '█' * (n // 5)
    print(f"  {icon} {cat[:35]:<35} {n:>4} |{bar}")

# גרף
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = [CATEGORIES[c]['color'] for c in counts.index]

# עמודות
bars = ax1.bar(range(len(counts)), counts.values,
               color=colors, alpha=0.85, edgecolor='white', linewidth=2)
ax1.set_xticks(range(len(counts)))
ax1.set_xticklabels([c.replace(' ', '\n') for c in counts.index], fontsize=10)
ax1.set_ylabel('מספר ערכים', fontsize=12)
ax1.set_title('כמות ערכים לפי קטגוריה', fontsize=14, fontweight='bold')
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             str(val), ha='center', va='bottom', fontsize=13, fontweight='bold')

# עוגה
ax2.pie(counts.values, labels=counts.index, colors=colors,
        autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 10})
ax2.set_title('חלוקה יחסית', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('01_category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 הגרף נשמר: 01_category_distribution.png")

## חלק ג: תיאוריית רשתות – Network Analysis

### מה זה גרף / רשת?
**גרף** הוא מבנה מתמטי המורכב מ:
- **צמתים** (Nodes): הישויות – כאן, ערכי ויקיפדיה
- **קשתות** (Edges): הקשרים – קישורים בין ערכים

### מדדי מרכזיות:

| מדד | הסבר | שימוש |
|-----|-------|-------|
| **In-Degree** | כמה ערכים מקשרים לכאן | "כמה פופולרי" |
| **Out-Degree** | כמה ערכים הערך מקשר אליהם | "כמה מחובר" |
| **PageRank** | אלגוריתם גוגל – חשיבות יחסית | "כמה סמכותי" |
| **Betweenness** | כמה פעמים על נתיב קצר | "גשר בין קבוצות" |

### גרף מכוון לעומת לא מכוון:
- **מכוון (Directed)**: קישור מ-א ל-ב ≠ קישור מ-ב ל-א
- **לא מכוון**: כל קישור הוא דו-כיווני

ויקיפדיה = **גרף מכוון** (ערך א' מקשר לב', אבל לא בהכרח ב' מקשר לא').

📖 **מומלץ**: Barabási, A.L. (2016). *Network Science*. (חינם: networksciencebook.com)

In [ ]:
# ============================================================
# בניית רשת הידע
# ============================================================

print("🕸️  בונה רשת ידע...")
print("=" * 55)

# גרף מכוון – קישורים בוויקיפדיה הם מכוונים (א → ב)
G = nx.DiGraph()

# כל כותרות הערכים שלנו (לסינון קישורים)
all_titles = set(df['title'].tolist())

# ── שלב 1: הוספת צמתים ──
print("📌 מוסיף צמתים...")
for _, row in df.iterrows():
    cat = row['category']
    G.add_node(
        row['title'],
        category=cat,
        color=CATEGORIES[cat]['color'],
        group=CATEGORIES[cat]['group'],
        pageid=str(row['pageid'])
    )
print(f"   {G.number_of_nodes()} צמתים נוספו")

# ── שלב 2: שליפת קישורים ──
print(f"\n🔗 שולף קישורים עבור {len(all_titles)} ערכים...")
print("   (שולף רק קישורים בין ערכים ברשימה שלנו)")

with tqdm(total=len(all_titles), desc="שליפת קישורים") as pbar:
    for title in list(all_titles):
        links = wiki.get_page_links(title, filter_titles=all_titles)
        for link in links:
            if link != title:
                G.add_edge(title, link)
        pbar.update(1)

print(f"\n✅ הרשת נבנתה!")
print(f"   צמתים : {G.number_of_nodes():,}")
print(f"   קשתות : {G.number_of_edges():,}")
print(f"   צפיפות: {nx.density(G):.4f}")

In [ ]:
# ============================================================
# ניתוח סטטיסטי – מדדי מרכזיות
# ============================================================

print("📊 מחשב מדדי מרכזיות...")

in_deg  = dict(G.in_degree())
out_deg = dict(G.out_degree())
pagerank = nx.pagerank(G, alpha=0.85)

# הוספה כתכונות צמתים
for n in G.nodes():
    G.nodes[n]['in_degree']  = in_deg.get(n, 0)
    G.nodes[n]['out_degree'] = out_deg.get(n, 0)
    G.nodes[n]['degree']     = in_deg.get(n, 0) + out_deg.get(n, 0)
    G.nodes[n]['pagerank']   = round(pagerank.get(n, 0), 6)

# טבלת נתונים
stats = []
for n in G.nodes():
    nd = G.nodes[n]
    stats.append({
        'כותרת':            n,
        'קטגוריה':          nd.get('category', ''),
        'קישורים_נכנסים':   nd['in_degree'],
        'קישורים_יוצאים':   nd['out_degree'],
        'סך_קישורים':       nd['degree'],
        'PageRank':         nd['pagerank']
    })

df_stats = pd.DataFrame(stats).sort_values('PageRank', ascending=False).reset_index(drop=True)
df_stats.to_csv('02_network_stats.csv', index=False, encoding='utf-8-sig')

print("\n⭐ 15 הערכים המרכזיים ביותר (PageRank):")
print("-" * 70)
for i, row in df_stats.head(15).iterrows():
    icon = CATEGORIES.get(row['קטגוריה'], {}).get('icon', '•')
    print(f"  {i+1:2}. {icon} {row['כותרת'][:42]:<42} PR={row['PageRank']:.5f}")

isolated = [n for n in G.nodes() if G.degree(n) == 0]
print(f"\n🔴 ערכים מבודדים (אין קישורים לאחרים): {len(isolated)}")

print("\n💾 סטטיסטיקות נשמרו: 02_network_stats.csv")

## חלק ד: ויזואליזציה אינטראקטיבית עם PyVis

### כיצד לקרוא את הרשת:
| אלמנט | משמעות |
|-------|----------|
| **גודל הצומת** | PageRank – ערכים גדולים = חשובים יותר |
| **צבע** | קטגוריה: 🔴 ארכיאולוגיה / 🔵 היסטוריה / 🟢 DH |
| **קשתות כתומות** | קישורים **בין** קטגוריות – מעניינים במיוחד! |
| **קשתות אפורות** | קישורים בתוך אותה קטגוריה |

### ניווט בויזואליזציה:
- 🖱️ **גלגל עכבר** = זום פנימה/חוצה
- 🖱️ **גרירה** = הזזת צמתים
- 👆 **לחיצה** = הדגשת הצומת והקשרים שלו
- 🖱️ **hover** = מידע מפורט על הצומת

In [ ]:
# ============================================================
# ויזואליזציה אינטראקטיבית – PyVis
# ============================================================

print("🎨 יוצר ויזואליזציה אינטראקטיבית...")

net = Network(
    height='720px',
    width='100%',
    bgcolor='#16213e',
    font_color='#ecf0f1',
    directed=True
)

net.set_options("""
{
  "physics": {
    "solver": "forceAtlas2Based",
    "forceAtlas2Based": {
      "gravitationalConstant": -60,
      "centralGravity": 0.005,
      "springLength": 120,
      "springConstant": 0.06,
      "damping": 0.5
    },
    "stabilization": {"iterations": 200, "fit": true}
  },
  "nodes": {
    "font": {"size": 12, "face": "Arial"},
    "borderWidth": 2,
    "borderWidthSelected": 4
  },
  "edges": {
    "smooth": {"type": "continuous"},
    "color": {"inherit": false},
    "arrows": {"to": {"enabled": true, "scaleFactor": 0.4}}
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 150,
    "navigationButtons": true,
    "keyboard": true
  }
}
""")

max_pr = max(G.nodes[n]['pagerank'] for n in G.nodes()) if G.nodes() else 1

for node in G.nodes():
    nd    = G.nodes[node]
    pr    = nd.get('pagerank', 0)
    cat   = nd.get('category', '')
    color = nd.get('color', '#95a5a6')
    size  = 10 + (pr / max_pr) * 40 if max_pr > 0 else 15
    
    tooltip = (
        f"<b>{node}</b><br>"
        f"<i>{cat}</i><br>"
        f"קישורים נכנסים: {nd.get('in_degree', 0)}<br>"
        f"קישורים יוצאים: {nd.get('out_degree', 0)}<br>"
        f"PageRank: {pr:.5f}"
    )
    
    net.add_node(
        node,
        label=node[:25],
        title=tooltip,
        color=color,
        size=size,
        group=nd.get('group', 1)
    )

for src, tgt in G.edges():
    cross = G.nodes[src].get('category', '') != G.nodes[tgt].get('category', '')
    net.add_edge(
        src, tgt,
        color='#f39c12' if cross else '#5d6d7e',
        width=2   if cross else 1,
        title='קשר בין-קטגוריאלי' if cross else ''
    )

net.write_html('03_knowledge_network.html')
print("✅ הרשת נשמרה: 03_knowledge_network.html")
print()

with open('03_knowledge_network.html', 'r', encoding='utf-8') as f:
    html_content = f.read()
display(HTML(html_content))

## חלק ה: ניתוח פערים – מה חסר בוויקיפדיה?

### שיטות לזיהוי פערים:

1. **קישורים אדומים (Red Links)**: ערכים שמוזכרים בטקסט אך לא קיימים
2. **רשימת מומחים**: נושאים חשובים לתחום שלא מתועדים
3. **ניתוח הרשת**: אזורים דלילים, ערכים מבודדים

### 💡 הפרויקט שלך:
כל ערך חסר = **פרויקט פוטנציאלי**!  
ניתן לכתוב ערך חדש, לשפר ערך קיים, או לתרגם מוויקיפדיה אחרת.

> *"The sum of all human knowledge"* – הסיסמה של ויקיפדיה  
> אבל מה **לא** נמצא שם עדיין?

In [ ]:
# ============================================================
# ניתוח פערים – קישורים אדומים
# ============================================================

print("🔴 מחפש קישורים אדומים (ערכים חסרים)...")
print("בודק את 15 הערכים המרכזיים ביותר...")
print()

top_pages      = df_stats.head(15)['כותרת'].tolist()
red_link_data  = []

for title in top_pages:
    missing = wiki.get_red_links(title)
    cat     = G.nodes[title].get('category', '') if title in G.nodes() else ''
    icon    = CATEGORIES.get(cat, {}).get('icon', '•')
    
    print(f"  {icon} {title[:38]:<38} → {len(missing)} קישורים אדומים")
    for m in missing[:5]:
        red_link_data.append({
            'ערך_חסר':  m,
            'מוזכר_ב': title,
            'קטגוריה':  cat
        })

if red_link_data:
    df_red      = pd.DataFrame(red_link_data)
    top_missing = df_red.groupby('ערך_חסר').size().sort_values(ascending=False)
    
    print(f"\n📊 ערכים חסרים עם הכי הרבה אזכורים:")
    print("-" * 55)
    for topic, count in top_missing.head(15).items():
        print(f"  ❌ {topic[:48]:<48} ({count}×)")
    
    df_red.to_csv('04_red_links.csv', index=False, encoding='utf-8-sig')
    print("\n💾 נשמר: 04_red_links.csv")
else:
    print("\n⚠️  לא נמצאו קישורים אדומים בדגימה זו")

In [ ]:
# ============================================================
# הצעות לפרויקטים – בדיקת נושאים חסרים
# ============================================================

SUGGESTED_PROJECTS = {
    'ארכיאולוגיה של ארץ ישראל': [
        'ארכיאולוגיה תת-ימית בישראל',
        'ארכיאו-בוטניקה בחפירות ישראל',
        'ניתוח איזוטופים בארכיאולוגיה',
        'ארכיאולוגיה דיגיטלית בישראל',
        'ארכיאולוגיה של הנוף בארץ ישראל',
        'ארכיאולוגיה ביזנטית בנגב',
        'ארכיאולוגיה של ימי הביניים בארץ ישראל',
        'LiDAR בארכיאולוגיה ישראלית',
        'ניהול מורשת תרבותית בישראל',
        'ארכיאולוגיה של מסחר עתיק בארץ ישראל',
    ],
    'היסטוריה של עם ישראל': [
        'היסטוריוגרפיה יהודית מודרנית',
        'יהדות אתיופיה – היסטוריה',
        'יהדות תימן – ההגירה לישראל',
        'יהודי אמריקה הלטינית – היסטוריה',
        'תולדות הדפוס העברי',
        'נשים בהיסטוריה היהודית',
        'יהדות איראן בעת החדשה',
        'יהודי מרוקו בישראל',
        'ספרות יהודית-ערבית',
        'יהדות הודו – קהילות ופזורה',
    ],
    'מדעי הרוח הדיגיטליים': [
        'עיבוד שפה טבעית לעברית',
        'דיגיטציה של כתבי יד עבריים',
        'GIS בארכיאולוגיה',
        'Wikidata לחוקרי מדעי הרוח',
        'Open Access במחקר ישראלי',
        'ניתוח רשת בהיסטוריה',
        'בינה מלאכותית בפרשנות טקסטים',
        'מפות דיגיטליות לחקר ארץ ישראל',
        'הגניזה הקהירית – פרויקטים דיגיטליים',
        'OCR לכתב יד עברי',
    ]
}

print("🎯 בודק אילו נושאים מוצעים חסרים בוויקיפדיה...")
print("=" * 65)

project_list = []
for category, topics in SUGGESTED_PROJECTS.items():
    print(f"\n{CATEGORIES[category]['icon']}  {category}:")
    
    existence = wiki.check_exists(topics)
    
    for topic in topics:
        exists = existence.get(topic, False)
        status = "✅ קיים  " if exists else "❌ חסר →"
        note   = " ← הזדמנות לפרויקט!" if not exists else ""
        print(f"  {status} {topic:<48}{note}")
        
        project_list.append({
            'נושא':              topic,
            'קטגוריה':           category,
            'קיים_בוויקיפדיה':   exists,
            'הצעה_לפרויקט':      not exists
        })
    
    time.sleep(0.5)

df_proj = pd.DataFrame(project_list)
missing_n = (~df_proj['קיים_בוויקיפדיה']).sum()

print(f"\n{'='*65}")
print(f"📊 סיכום: {missing_n} מתוך {len(df_proj)} נושאים חסרים בוויקיפדיה")
print(f"   → {missing_n} הזדמנויות לפרויקטי סיום!")

df_proj.to_csv('05_project_suggestions.csv', index=False, encoding='utf-8-sig')
print("\n💾 נשמר: 05_project_suggestions.csv")

In [ ]:
# ============================================================
# שמירת כל הנתונים
# ============================================================

print("💾 שומר נתונים ועצמי גרף...")

df.to_csv('07_all_pages.csv', index=False, encoding='utf-8-sig')
df_stats.to_csv('02_network_stats.csv', index=False, encoding='utf-8-sig')

# קבצי גרף לניתוח בכלים חיצוניים (Gephi, Cytoscape)
nx.write_gexf(G, '08_network.gexf')
nx.write_graphml(G, '09_network.graphml')

print("\n📁 קבצים שנוצרו:")
for fname in sorted(os.listdir('.')):
    if any(fname.endswith(ext) for ext in ['.csv', '.html', '.gexf', '.graphml', '.png']):
        size = os.path.getsize(fname)
        print(f"   📄 {fname:<42} {size:>10,} bytes")

print("\n🎉 מחברת 1 הושלמה!")
print()
print("📚 המשך:")
print("  1️⃣  עיינו ב-05_project_suggestions.csv – בחרו נושא")
print("  2️⃣  פתחו את 03_knowledge_network.html – חקרו את הרשת")
print("  3️⃣  עברו למחברת 2: קריאה מרחוק! →")

## 📝 תרגילים

### תרגיל 1 – בסיסי ⭐
בחרו קטגוריה שמעניינת אתכם ובצעו ניתוח דומה:
```python
# שנו לקטגוריה שבחרתם:
pages = wiki.get_category_members("ארכיאולוגיה של ים המלח")
```

### תרגיל 2 – בינוני ⭐⭐
הוסיפו לויזואליזציה **מקרא (Legend)** המסביר את הצבעים, הצורות, וגודל הצמתים.  
*רמז: PyVis תומך ב-HTML מותאם אישית בתיאור הרשת.*

### תרגיל 3 – מתקדם ⭐⭐⭐
השוו את הרשת **העברית** לרשת **האנגלית** לאותן קטגוריות.  
שנו את `BASE_URL` ל-`https://en.wikipedia.org/w/api.php` וחפשו:  
- אילו ערכים קיימים באנגלית אבל לא בעברית?

---

## 🔗 משאבים
- [MediaWiki API Documentation](https://www.mediawiki.org/wiki/API:Main_page)
- [NetworkX Documentation](https://networkx.org/documentation/stable/)
- [PyVis Documentation](https://pyvis.readthedocs.io/)
- [Network Science (Barabási, חינם)](http://networksciencebook.com/)
- [Programming Historian](https://programminghistorian.org/) – שיעורי DH